# 7Learnings Data Scientist Challenge: Retail Demand Forecasting

Welcome to the **7Learnings Data Scientist Case Study**!

## Business Context
At 7Learnings, our predictive machine learning engines optimize retail pricing and supply chain planning for leading enterprise brands. Accurate demand forecasting is the critical foundation: an under-forecast leads to stockouts, lost revenue, and damaged customer loyalty; an over-forecast leads to bloated working capital, warehouse congestion, and aggressive end-of-season markdown depreciation.

## Problem Statement
Your mission is to forecast daily product-level unit sales (`sales_before_returns` aggregated at `(product_id, date)` grain) across all catalog products for the **14-day horizon**:

$$\mathbf{Forecast\ Horizon:\ 2026\text{-}09\text{-}07\ \text{through}\ 2026\text{-}09\text{-}20\ (inclusive)}$$

---

## ⚠️ Critical Rule: Historical Cutoff & Temporal Leakage Prevention
- **Training Cutoff Timestamp:** `2026-09-06 23:59:59`
- While `7ldata.duckdb` contains transaction logs extending into September 2026, you **must strictly train models and engineer features using ONLY observations on or before `2026-09-06`**.
- In production, future ground truth does not exist. Any feature engineered with lookahead leakage over the forecast horizon will lead to disqualification.
- Post-cutoff transactions are reserved exclusively for our internal evaluation benchmark.

---
## 🤖 AI Usage & Transparency Declaration

At 7Learnings, we embrace generative AI tools (ChatGPT, Claude, GitHub Copilot, Cursor, etc.) as everyday accelerators. **You are fully permitted to utilize AI assistants.**

However, we expect full transparency and deep technical ownership. Please complete the disclosure declaration below before starting:

```markdown
### Candidate AI Disclosure Form
- AI Assistants Used: [e.g. None / Claude / ChatGPT-4o / GitHub Copilot / Cursor]
- Primary Tasks Assisted by AI: [e.g. Boilerplate syntax, SQL query drafting, exploratory plot formatting]
- Key Prompts / Queries Used: [Brief summary or 1-2 example prompts]
- Ownership Confirmation: I have independently reviewed and verified all code and explanations. I understand every mathematical and architectural choice and am fully prepared to explain and defend them during the technical interview.
```

---
## 0. Environment Setup & Data Inspection

### Database Decompression (if needed)
If `7ldata.duckdb` is not present in your local directory, unpack it from the compressed archive `7ldata.duckdb.zst`:
```bash
# Optional download if setting up in fresh cloud instance:
# gcloud storage cp gs://7learnings-candidate-datasets/intermediate/7ldata.duckdb.zst .

# Decompress via zstd command line:
zstd -d 7ldata.duckdb.zst -o 7ldata.duckdb
```

Let us initialize our environment and connect to `7ldata.duckdb`.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import sklearn
import datetime

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print(f"DuckDB version:   {duckdb.__version__}")
print(f"Pandas version:   {pd.__version__}")
print(f"XGBoost version:  {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")

In [ ]:
# Connect to the local DuckDB database
con = duckdb.connect("7ldata.duckdb", read_only=True)

# Inspect all tables and row counts
tables = con.execute("SHOW TABLES").fetchall()
print("Available tables in 7ldata.duckdb:")
for (tbl,) in tables:
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  - {tbl:<22}: {cnt:>10,d} rows")

In [ ]:
# Inspect the schema and sample rows of transactions
con.execute("DESCRIBE transactions").df()

---
## Part 1: Data Quality Audit & Cleaning

### Context
The `transactions` table has arrived directly from raw upstream retail logs and contains data quality anomalies. Before aggregating and modeling, you must audit the dataset, identify any anomalies, document your findings, and apply appropriate cleaning steps.


In [ ]:
# TODO: Audit data quality issues in transactions
# Inspect distributions, check relationships, and identify anomalies

In [ ]:
# TODO: Implement cleaning transformations (in DuckDB SQL or Pandas)
# Create a clean transaction dataset up to the cutoff date (2026-09-06)

---
## Part 2: Feature Engineering & Aggregation

### Context
Our forecasting target is daily product-level unit sales (`sales_before_returns`).

### Guidelines
- Aggregate transactions to daily product-level sales (`(product_id, date)`).
- You are free to engineer any features you see fit for your model using the provided tables (`product_attributes`, `promotions`, `stock`, `traffic`, and cleaned transactions).
- Ensure all feature engineering strictly respects the training cutoff (`<= 2026-09-06`) to prevent lookahead leakage.
- *Tip: DuckDB SQL allows fast in-database aggregation and joins before pulling into Pandas.*

In [ ]:
# TODO: Aggregate daily product-level sales up to 2026-09-06
# Engineer your feature set from the available tables

---
## Part 3: Predictive Modeling

### Context
Train a predictive model to forecast daily product sales across the 14-day horizon (`2026-09-07` through `2026-09-20`).

### Modeling Strategy Considerations
- **Model Choice:** You are free to use any model architecture or auditable library of your choice, provided you supply reproducible code and any extra dependencies needed to run it. XGBoost is pre-installed in the environment as a recommended starting point.
- **Validation Design:** Establish a sound time-based validation split to evaluate your model locally.
- **Multi-Day Horizon Strategy:** Formulate a clear strategy for multi-day forecasting over the 14-day period.

In [ ]:
# TODO: Set up temporal train/validation split
# Define feature matrix X and target y (sales_before_returns)

In [ ]:
# TODO: Train forecasting model (XGBoost / LightGBM)
# Evaluate on validation holdout

---
## Part 4: Model Evaluation & Business Insights

### 1. Quantitative Evaluation
Evaluate your validation forecasts using standard retail metrics:
- **WAPE (Weighted Absolute Percentage Error):**
  $$\text{WAPE} = \frac{\sum_{i} |y_i - \hat{y}_i|}{\sum_i y_i}$$
- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Squared Error)**
- **Forecast Bias (%):**
  $$\text{Bias} = \frac{\sum_i \hat{y}_i - \sum_i y_i}{\sum_i y_i}$$

### 2. Feature Importance & Interpretability
Extract and visualize feature importances. Which features drive the demand predictions most strongly?

### 3. Business Reflection
- What are the commercial tradeoffs between an over-forecast vs an under-forecast in retail pricing and replenishment?
- If a product is out of stock in historical data (`stock == 0`), how does that affect observed sales, and how should a demand forecasting model account for censored demand?

In [ ]:
# TODO: Compute validation metrics (WAPE, MAE, RMSE, Bias)
# Plot feature importances and predicted vs actual sales

### Business Insights & Operational Takeaways
*[Write your business insights, metric interpretations, and recommendations here]*

---
## Part 5: Generate Submission Predictions (`predictions.csv`)

Generate your final predictions for the 14-day prediction period (`2026-09-07` through `2026-09-20`) across active catalog products.

### Format Specifications
- **File:** `predictions.csv`
- **Columns:** `product_id, date, predicted_sales`
- **Prediction Period:** `2026-09-07` through `2026-09-20` (14 days)
- **Date Format:** `YYYY-MM-DD`
- **SKU Normalization:** Cleaned string format without whitespace padding; bundle SKUs formatted with lowercase `_x_`.

In [ ]:
# TODO: Generate predictions for 2026-09-07 through 2026-09-20
# Expected DataFrame format:
# submission_df = pd.DataFrame({
#     "product_id": ...,
#     "date": ...,
#     "predicted_sales": ...
# })

# Pre-Submission Validation Checks:
# 1. Required columns
# assert set(submission_df.columns) == {"product_id", "date", "predicted_sales"}
#
# 2. No nulls, NaNs, or negative predictions
# assert submission_df["predicted_sales"].isna().sum() == 0, "NaNs found in predictions!"
# assert (submission_df["predicted_sales"] >= 0).all(), "Negative predictions detected!"
#
# 3. SKU normalization (whitespace trimmed, lowercase bundle marker)
# assert (submission_df["product_id"] == submission_df["product_id"].str.strip()).all(), "Leading/trailing whitespace in product_id!"
# assert (~submission_df["product_id"].str.contains("_X_")).all(), "Bundle SKUs must use lowercase '_x_' matching catalog metadata!"
#
# 4. Dates
# assert submission_df["date"].min() == "2026-09-07" and submission_df["date"].max() == "2026-09-20", "Date range mismatch!"
#
# Export:
# submission_df.to_csv("predictions.csv", index=False)
# print("predictions.csv successfully generated and passed all validation checks!")